In [1]:
import wandb
api = wandb.Api()
import numpy as np

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
runs = api.runs("ramithx1/Batch-Run-Maximum-Parsimony-A100-META-optimized")

In [4]:
errors = {}

sankoffs = {}
elves = {}

for l in [4,8,16,32,64,128]:
    errors[str(l)] = []
    sankoffs[str(l)] = []
    elves[str(l)] = []
    
    for run in runs:
        if(run.config['n_leaves'] == l and run.config['tLs']==[0, 0.005, 10, 50]):
            
            if("custom" in run.name or 'best cost' not in run.summaryMetrics.keys()):
                continue
                
            if("Final" not in run.name):
                continue
            
            # print(run.config['epochs'])
            
            cost = run.summaryMetrics['best cost'] #run.summaryMetrics['traversal cost']
            gt   = run.summaryMetrics['sankoff_cost'] 


            errors[str(l)].append(cost - gt)
            sankoffs[str(l)].append(gt)
            elves[str(l)].append(cost)
            

            # print(run.summaryMetrics.keys())
            # print(run.summaryMetrics['sankoff_cost'])
            # print(run.summaryMetrics['traversal cost'])
            # print(run.history)
        


In [5]:
sankoffs

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [652, 656, 656, 650, 648, 646, 668, 645, 662, 648],
 '16': [1404, 1413, 1422, 1398, 1399, 1402, 1421, 1401, 1420, 1396],
 '32': [2913, 2908, 2932, 2907, 2916, 2918, 2916, 2911, 2930, 2903],
 '64': [5927, 5899, 5954, 5948, 5947, 5926, 5929, 5896, 5929, 5938],
 '128': [11963, 11980, 12011, 11966, 11980, 11960, 11968, 11979, 11941, 11963]}

In [6]:
errors

{'4': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 '8': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 '16': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 '32': [22, 16, 21, 36, 20, 10, 20, 15, 17, 32],
 '64': [277, 216, 267, 245, 279, 316, 228, 265, 238, 262],
 '128': [945, 839, 1054, 1044, 1042, 765, 811, 837, 883, 924]}

In [7]:
elves

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [652, 656, 656, 650, 648, 646, 668, 645, 662, 648],
 '16': [1404, 1413, 1423, 1398, 1399, 1402, 1421, 1401, 1420, 1396],
 '32': [2935, 2924, 2953, 2943, 2936, 2928, 2936, 2926, 2947, 2935],
 '64': [6204, 6115, 6221, 6193, 6226, 6242, 6157, 6161, 6167, 6200],
 '128': [12908, 12819, 13065, 13010, 13022, 12725, 12779, 12816, 12824, 12887]}

In [8]:
for key in [4,8,16,32,64,128]:

    err = np.array(errors[str(key)])
    s_gt = np.array(sankoffs[str(key)])
    sol  = np.array(elves[str(key)])
    
    print(key, end = " -> ")
    assert err.shape == sol.shape
    assert s_gt.shape == sol.shape
    print(f" average sankoff = {s_gt.mean()}, avg ans = {sol.mean()}, avg error = {err.mean()}, std = {err.std()}, % error = {err.mean()/s_gt.mean()*100}") 

    

4 ->  average sankoff = 277.2, avg ans = 277.2, avg error = 0.0, std = 0.0, % error = 0.0
8 ->  average sankoff = 653.1, avg ans = 653.1, avg error = 0.0, std = 0.0, % error = 0.0
16 ->  average sankoff = 1407.6, avg ans = 1407.7, avg error = 0.1, std = 0.30000000000000004, % error = 0.007104290991759024
32 ->  average sankoff = 2915.4, avg ans = 2936.3, avg error = 20.9, std = 7.39526875779373, % error = 0.716882760513137
64 ->  average sankoff = 5929.3, avg ans = 6188.6, avg error = 259.3, std = 27.437383257154828, % error = 4.373197510667364
128 ->  average sankoff = 11971.1, avg ans = 12885.5, avg error = 914.4, std = 99.63352849317342, % error = 7.638395803226103
